# Submit Workbook Goblins To The Repository

Run this as a normal JupyterHub user with access to the Goblin King project.

In [ ]:
import importlib
import os
import site
import subprocess
import sys
from pathlib import Path

package = (
    os.environ.get("GOBLIN_KING_REPOSITORY_NOTEBOOK_PACKAGE")
    or os.environ.get("GOBLIN_KING_NOTEBOOK_PACKAGE")
    or "git+https://github.com/tashabits/goblin-king.git"
)
print(f"Installing notebook helper from {package}")
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--disable-pip-version-check",
    "--quiet",
    "--user",
    "--force-reinstall",
    "--no-deps",
    package,
])
user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.insert(0, user_site)
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "goblin_king" or module_name.startswith("goblin_king."):
        del sys.modules[module_name]

import goblin_king  # noqa: E402, I001
from goblin_king.notebooks import GoblinKingNotebookClient  # noqa: E402, I001

token = os.environ.get("JUPYTERHUB_API_TOKEN") or os.environ.get("GOBLIN_KING_API_TOKEN")
if not token:
    raise RuntimeError("JUPYTERHUB_API_TOKEN or GOBLIN_KING_API_TOKEN is required")

client = GoblinKingNotebookClient(
    api_url=os.environ.get(
        "GOBLIN_KING_API_URL",
        "http://goblin-king-api.default.svc.cluster.local:8000",
    ),
    repository_url=os.environ.get("GOBLIN_KING_REPOSITORY_URL") or None,
    token=token,
    request_timeout_seconds=180,
)
print(f"Loaded goblin_king from {Path(goblin_king.__file__).resolve()}")
print(f"API: {client.api_url}")
print(f"Repository: {client.repository_url or client.api_url}")

In [ ]:
def repository_workbook_hello(payload):
    name = payload.get("name", "Repository")
    return {
        "message": f"Hello {name}",
        "source": "repository-submit-workbook",
    }

function_submission = client.submit_repository_function(
    repository_workbook_hello,
    name=os.environ.get("GOBLIN_REPOSITORY_FUNCTION_NAME", "workbook.shared-hello"),
    display_name="Workbook Shared Hello",
    description="Short hello-world function submitted from a notebook",
    tags=["workbook", "hello"],
    timeout_seconds=30,
)
function_submission.latest_response

In [ ]:
function_validation = function_submission.validate(
    {"name": "Validation"},
    progress=True,
)
function_review = function_submission.request_review(
    "Function validated from the submitter workbook",
    progress=True,
)
{
    "entry_id": function_submission.entry_id,
    "name": function_submission.name,
    "status": function_submission.entry["status"],
    "version_status": function_submission.version["status"],
    "validation": function_validation["validation"],
}

In [ ]:
SERVICE_SOURCE = """
from fastapi import FastAPI

app = FastAPI()

@app.get("/hello")
def hello():
    return {
        "message": "Hello World",
        "source": "repository-submit-workbook-service",
    }
""".strip()

service_submission = client.submit_repository_service(
    source=SERVICE_SOURCE,
    name=os.environ.get("GOBLIN_REPOSITORY_SERVICE_NAME", "workbook.shared-long-hello"),
    app_name="app",
    requirements=["fastapi>=0.115,<1"],
    probe_path="/hello",
    display_name="Workbook Shared Long Hello",
    description="ASGI hello-world service submitted from a notebook",
    tags=["workbook", "service", "hello"],
)
service_submission.latest_response

In [ ]:
service_validation = service_submission.validate(progress=True, timeout_seconds=180)
service_review = service_submission.request_review(
    "Service validated from the submitter workbook",
    progress=True,
)
{
    "entry_id": service_submission.entry_id,
    "name": service_submission.name,
    "status": service_submission.entry["status"],
    "version_status": service_submission.version["status"],
    "validation": service_validation["validation"],
}